# Disease embeddings on Colab GPU (T4)

1. On your PC run `python -m src.ingestion.export_diseases_for_colab` (after `load_diseases`).
2. **Runtime → Change runtime type → GPU** (often T4).
3. Upload `diseases_to_embed.parquet` (next cell).
4. Run all cells; download `disease_embeddings.parquet`.
5. On your PC save it as `data/processed/disease_embeddings.parquet` and run `python -m src.ingestion.import_embeddings_from_parquet`.

In [ ]:
!nvidia-smi

In [ ]:
%pip install -q sentence-transformers pandas pyarrow tqdm

In [ ]:
import torch
assert torch.cuda.is_available(), "Enable GPU: Runtime → Change runtime type → GPU"
print("CUDA:", torch.cuda.get_device_name(0))

In [ ]:
from google.colab import files
import shutil
from pathlib import Path

IN_PATH = Path("/content/diseases_to_embed.parquet")
if not IN_PATH.is_file():
    print("Upload diseases_to_embed.parquet from your PC (export step):")
    up = files.upload()
    name = next(iter(up.keys()))
    shutil.move(name, IN_PATH)
print("Input:", IN_PATH, "size", IN_PATH.stat().st_size)

In [ ]:
import pandas as pd
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer

MODEL = "sentence-transformers/all-MiniLM-L6-v2"
ENCODE_BATCH = 512

df = pd.read_parquet(IN_PATH)
assert "id" in df.columns and "name" in df.columns, df.columns.tolist()
names = df["name"].astype(str).tolist()
ids = df["id"].astype(str).tolist()

model = SentenceTransformer(MODEL, device="cuda")
all_emb = []
for i in tqdm(range(0, len(names), ENCODE_BATCH)):
    batch = names[i : i + ENCODE_BATCH]
    vecs = model.encode(
        batch,
        batch_size=min(128, len(batch)),
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    all_emb.extend([v.tolist() for v in vecs])

out_df = pd.DataFrame({"id": ids, "embedding": all_emb})
assert len(out_df) == len(df)
assert len(out_df.iloc[0]["embedding"]) == 384

OUT = Path("/content/disease_embeddings.parquet")
out_df.to_parquet(OUT, index=False)
print("Wrote", OUT, OUT.stat().st_size)

In [ ]:
from google.colab import files
files.download("/content/disease_embeddings.parquet")